In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error



In [2]:
users = pd.read_csv('/datasets/users_behavior.csv')

In [3]:
print(users.shape)

(3214, 5)


In [4]:
print(users.head())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


In [5]:
# to split the data source, the validation is 20%, train is 80% and will have to use the train, validation and test data
train, validation = train_test_split(users, test_size =0.2)
print(train.shape)
print(validation.shape)

(2571, 5)
(643, 5)


In [6]:
# we need to split the validation into half
validation, test = train_test_split(validation, test_size =0.5)
print(validation.shape)
print(test.shape)


(321, 5)
(322, 5)


In [7]:

features_train = train.drop(['is_ultra'], axis = 1)
target_train = train['is_ultra']
features_validation = validation.drop(['is_ultra'], axis = 1)
target_validation = validation['is_ultra']
features_test = test.drop(['is_ultra'], axis = 1)
target_test = test['is_ultra']
print(features_train.shape)
print(target_train.shape)
print(features_validation.shape)
print(target_validation.shape)
print(features_test.shape)
print(target_test.shape)

(2571, 4)
(2571,)
(321, 4)
(321,)
(322, 4)
(322,)


In [8]:

model = DecisionTreeClassifier(random_state =12345)
model.fit(features_train, target_train)
score_decision_tree = model.score(features_validation, target_validation)
# testing the train model using another new data i.e validation dataset, note: features_validation does not change, it is the data given
print(score_decision_tree)

0.719626168224299


In [9]:
# using the overfitting pattern  range of 1 - 20  so as to see the full pattern of overfitting
depths = range(1,20)
for depth in range(1,20):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_validation = model.predict(features_validation)
    print("max_depth =", depth, ": ", end='')
    print(accuracy_score(target_validation, predictions_validation))



max_depth = 1 : 0.7507788161993769
max_depth = 2 : 0.794392523364486
max_depth = 3 : 0.8099688473520249
max_depth = 4 : 0.8037383177570093
max_depth = 5 : 0.7975077881619937
max_depth = 6 : 0.7975077881619937
max_depth = 7 : 0.8006230529595015
max_depth = 8 : 0.7975077881619937
max_depth = 9 : 0.8130841121495327
max_depth = 10 : 0.8099688473520249
max_depth = 11 : 0.7881619937694704
max_depth = 12 : 0.7975077881619937
max_depth = 13 : 0.7725856697819314
max_depth = 14 : 0.7725856697819314
max_depth = 15 : 0.778816199376947
max_depth = 16 : 0.7570093457943925
max_depth = 17 : 0.7694704049844237
max_depth = 18 : 0.7476635514018691
max_depth = 19 : 0.7414330218068536


The range 1-20 was choosen to know where the model performs best on the validation data. The max_depth is on 7 with accuracy of about 80.4%,

In [11]:
# using RandomForestClassifier
best_accuracyscore = 0
best_estimator = 0
for estimator in range(1, 20):
    model = RandomForestClassifier(random_state=12345, n_estimators=estimator)
    model.fit(features_train, target_train)
    score = model.score(features_validation, target_validation)
    if score > best_accuracyscore:
        best_accuracyscore = score
        best_estimator = estimator

print("Accuracy of the best model on the validation set (n_estimators = {}): {}".format(best_estimator, best_accuracyscore))

final_model = RandomForestClassifier(random_state=12345, n_estimators=best_estimator)
final_model.fit(features_train, target_train)

Accuracy of the best model on the validation set (n_estimators = 7): 0.8317757009345794


RandomForestClassifier(n_estimators=7, random_state=12345)

In [12]:
# logistic regression
model = LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train) 
score_train = model.score(features_train, target_train)  
score_validation = model.score(features_validation, target_validation)
print(
    "Accuracy of the logistic regression model on the training set:",
    score_train,
)
print(
    "Accuracy of the logistic regression model on the validation set:",
    score_validation,
)

Accuracy of the logistic regression model on the training set: 0.7078957604045119
Accuracy of the logistic regression model on the validation set: 0.6915887850467289


The training accuracy and validation accuracy are 70.44% and 70.40% respectively. They are almost identical. They result s are below the 75% accuracy for megaline

In [16]:
# Checking the quality of the model using the test set
# here the best model ( the Random Forest with 7 estimators would be used)
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)
print(f"Accuracy of the best model on the test set: {test_accuracy}")

Accuracy of the best model on the test set: 0.782608695652174


Getting 0.7826 which is 78.26% accuracy means Random Forest model correctly predicted the right model plan (smart or Ultra) for 78.26% customers in the test set.